# Mixed Precision Inference for LLMs

**Stage 1: Basic Optimization**

This notebook demonstrates mixed precision inference (FP16/BF16), which provides:
- **2x memory reduction** compared to FP32
- **1.5-2x speed improvement** on modern GPUs
- **Minimal quality loss** for most LLM tasks

## Precision Formats Explained

| Format | Bits | Range | Precision | Best For |
|--------|------|-------|-----------|----------|
| **FP32** | 32 | ±3.4×10³⁸ | 7 decimal digits | Training, high precision |
| **FP16** | 16 | ±65,504 | 3 decimal digits | Inference on Volta+ GPUs |
| **BF16** | 16 | ±3.4×10³⁸ | 2 decimal digits | Inference on Ampere+ GPUs |

### Key Differences
- **FP16**: More precision, smaller range (risk of overflow/underflow)
- **BF16**: Same range as FP32, less precision (more stable for large models)
- **Hardware**: BF16 requires Ampere (A100, RTX 30/40 series) or newer

**Reference**: See `LLM_INFERENCE_ROADMAP.md` - Stage 1, Basic Optimizations

In [ ]:
# Install required packages
!pip install torch transformers accelerate matplotlib pandas numpy -q

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from typing import Dict, List
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
    
    # Check BF16 support
    compute_capability = torch.cuda.get_device_capability(0)
    bf16_supported = compute_capability[0] >= 8  # Ampere or newer
    print(f"BF16 Support: {bf16_supported}")
else:
    bf16_supported = False
    print("Running on CPU - mixed precision benefits limited")

## 1. Understanding Precision Formats

Let's visualize the differences between FP32, FP16, and BF16.

In [ ]:
def show_precision_representation():
    """Visualize floating point representation."""
    
    formats = {
        'FP32': {'total': 32, 'sign': 1, 'exponent': 8, 'mantissa': 23},
        'FP16': {'total': 16, 'sign': 1, 'exponent': 5, 'mantissa': 10},
        'BF16': {'total': 16, 'sign': 1, 'exponent': 8, 'mantissa': 7},
    }
    
    fig, axes = plt.subplots(3, 1, figsize=(12, 6))
    
    for idx, (name, bits) in enumerate(formats.items()):
        ax = axes[idx]
        
        # Draw bit boxes
        total_bits = bits['total']
        colors = ['red'] * bits['sign'] + ['blue'] * bits['exponent'] + ['green'] * bits['mantissa']
        
        for i, color in enumerate(colors):
            ax.add_patch(plt.Rectangle((i, 0), 1, 1, fill=True, color=color, alpha=0.6, edgecolor='black'))
        
        # Labels
        ax.text(0.5, 0.5, 'S', ha='center', va='center', fontsize=10, fontweight='bold')
        ax.text(bits['sign'] + bits['exponent']/2, 0.5, 'Exponent', 
               ha='center', va='center', fontsize=10, fontweight='bold')
        ax.text(bits['sign'] + bits['exponent'] + bits['mantissa']/2, 0.5, 'Mantissa', 
               ha='center', va='center', fontsize=10, fontweight='bold')
        
        ax.set_xlim(0, total_bits)
        ax.set_ylim(0, 1)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f"{name}: {bits['total']} bits total (Sign: {bits['sign']}, Exp: {bits['exponent']}, Mantissa: {bits['mantissa']})", 
                    fontsize=12, fontweight='bold', pad=10)
    
    plt.tight_layout()
    plt.savefig('precision_formats.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nKey Insights:")
    print("1. BF16 has same exponent range as FP32 (8 bits) - better for large values")
    print("2. FP16 has more mantissa bits (10 vs 7) - better precision for small values")
    print("3. Both FP16 and BF16 use 2x less memory than FP32")

show_precision_representation()

In [ ]:
# Demonstrate precision and range differences
def compare_precision_characteristics():
    """Compare numerical characteristics of different precisions."""
    
    test_values = [1.0, 0.0001, 1000.0, 65000.0, 100000.0, 1e-8, 1e8]
    
    results = []
    
    for val in test_values:
        fp32_val = torch.tensor(val, dtype=torch.float32)
        fp16_val = torch.tensor(val, dtype=torch.float16)
        bf16_val = torch.tensor(val, dtype=torch.bfloat16)
        
        results.append({
            'Original': val,
            'FP32': fp32_val.item(),
            'FP16': fp16_val.item() if not torch.isinf(fp16_val) else 'inf',
            'BF16': bf16_val.item(),
            'FP16 Error': abs(fp32_val.item() - fp16_val.item()) if not torch.isinf(fp16_val) else 'overflow',
            'BF16 Error': abs(fp32_val.item() - bf16_val.item()),
        })
    
    df = pd.DataFrame(results)
    print("Precision Comparison Across Value Ranges:")
    print("="*100)
    print(df.to_string(index=False))
    
    print("\nObservations:")
    print("1. FP16 overflows at ~65,504 (limited exponent range)")
    print("2. BF16 handles large values like FP32 (same exponent range)")
    print("3. Both lose precision compared to FP32, but acceptable for inference")

compare_precision_characteristics()

## 2. Loading Models in Different Precisions

In [ ]:
def load_model_in_precision(model_name: str, dtype: torch.dtype, device: str = "cuda"):
    """Load model in specified precision."""
    print(f"Loading {model_name} in {dtype}...")
    
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    
    start = time.time()
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map=device if device == "cpu" else "auto",
        low_cpu_mem_usage=True,
    )
    model.eval()
    
    load_time = time.time() - start
    
    # Calculate memory
    param_count = sum(p.numel() for p in model.parameters())
    
    if device == "cuda":
        memory_allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        memory_reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    else:
        memory_allocated = 0
        memory_reserved = 0
    
    bytes_per_param = {
        torch.float32: 4,
        torch.float16: 2,
        torch.bfloat16: 2,
    }[dtype]
    
    theoretical_memory = param_count * bytes_per_param / (1024 ** 3)
    
    return model, {
        'load_time': load_time,
        'param_count': param_count,
        'memory_allocated_gb': memory_allocated,
        'memory_reserved_gb': memory_reserved,
        'theoretical_memory_gb': theoretical_memory,
        'dtype': str(dtype),
    }


# Load GPT-2 in different precisions
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Loading models in different precisions...\n")
print("="*80)

In [ ]:
# Load FP32 model
model_fp32, stats_fp32 = load_model_in_precision(model_name, torch.float32, device)
print(f"FP32 loaded in {stats_fp32['load_time']:.2f}s")
print(f"  Parameters: {stats_fp32['param_count']/1e6:.1f}M")
print(f"  Theoretical memory: {stats_fp32['theoretical_memory_gb']:.2f} GB")
if device == "cuda":
    print(f"  GPU memory allocated: {stats_fp32['memory_allocated_gb']:.2f} GB")
print()

# Clear memory before loading next
del model_fp32
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Load FP16 model
model_fp16, stats_fp16 = load_model_in_precision(model_name, torch.float16, device)
print(f"FP16 loaded in {stats_fp16['load_time']:.2f}s")
print(f"  Parameters: {stats_fp16['param_count']/1e6:.1f}M")
print(f"  Theoretical memory: {stats_fp16['theoretical_memory_gb']:.2f} GB")
if device == "cuda":
    print(f"  GPU memory allocated: {stats_fp16['memory_allocated_gb']:.2f} GB")
print(f"  Memory reduction vs FP32: {stats_fp32['theoretical_memory_gb'] / stats_fp16['theoretical_memory_gb']:.2f}x")
print()

del model_fp16
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Load BF16 model (if supported)
if bf16_supported or device == "cpu":
    model_bf16, stats_bf16 = load_model_in_precision(model_name, torch.bfloat16, device)
    print(f"BF16 loaded in {stats_bf16['load_time']:.2f}s")
    print(f"  Parameters: {stats_bf16['param_count']/1e6:.1f}M")
    print(f"  Theoretical memory: {stats_bf16['theoretical_memory_gb']:.2f} GB")
    if device == "cuda":
        print(f"  GPU memory allocated: {stats_bf16['memory_allocated_gb']:.2f} GB")
    print(f"  Memory reduction vs FP32: {stats_fp32['theoretical_memory_gb'] / stats_bf16['theoretical_memory_gb']:.2f}x")
    
    del model_bf16
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
else:
    print("BF16 not supported on this GPU (requires Ampere or newer)")
    stats_bf16 = None

In [ ]:
# Visualize memory comparison
memory_stats = [stats_fp32, stats_fp16]
labels = ['FP32', 'FP16']

if stats_bf16:
    memory_stats.append(stats_bf16)
    labels.append('BF16')

fig, ax = plt.subplots(figsize=(10, 6))

theoretical_mem = [s['theoretical_memory_gb'] for s in memory_stats]
x = np.arange(len(labels))

bars = ax.bar(x, theoretical_mem, color=['coral', 'lightblue', 'lightgreen'][:len(labels)])

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, theoretical_mem)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.2f} GB\n({val/theoretical_mem[0]:.1f}x)',
            ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Memory (GB)', fontsize=12)
ax.set_title(f'Model Memory Usage by Precision ({model_name.upper()})', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('memory_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMemory Savings:")
print(f"FP16 vs FP32: {(1 - stats_fp16['theoretical_memory_gb']/stats_fp32['theoretical_memory_gb'])*100:.1f}% reduction")
if stats_bf16:
    print(f"BF16 vs FP32: {(1 - stats_bf16['theoretical_memory_gb']/stats_fp32['theoretical_memory_gb'])*100:.1f}% reduction")

## 3. Speed Benchmarking: FP32 vs FP16 vs BF16

In [ ]:
def benchmark_inference(model, tokenizer, prompt: str, num_tokens: int = 50, num_runs: int = 5):
    """Benchmark inference speed."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    
    # Warmup
    with torch.no_grad():
        _ = model.generate(input_ids, max_new_tokens=10, use_cache=True)
    
    if device == "cuda":
        torch.cuda.synchronize()
    
    times = []
    for _ in range(num_runs):
        start = time.time()
        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                max_new_tokens=num_tokens,
                use_cache=True,
                do_sample=False,
            )
        
        if device == "cuda":
            torch.cuda.synchronize()
        
        times.append(time.time() - start)
    
    avg_time = np.mean(times)
    std_time = np.std(times)
    tokens_per_sec = num_tokens / avg_time
    
    return {
        'avg_time': avg_time,
        'std_time': std_time,
        'tokens_per_sec': tokens_per_sec,
        'times': times,
    }


test_prompts = [
    "The future of artificial intelligence",
    "In a world where technology advances rapidly",
    "The history of computing began with mechanical calculators",
]

num_tokens_generate = 100
print(f"Running speed benchmarks ({num_tokens_generate} tokens per prompt)...\n")
print("="*80)

In [ ]:
# Benchmark FP32
print("\nBenchmarking FP32...")
model_fp32, _ = load_model_in_precision(model_name, torch.float32, device)

fp32_results = []
for i, prompt in enumerate(test_prompts):
    result = benchmark_inference(model_fp32, tokenizer, prompt, num_tokens=num_tokens_generate)
    fp32_results.append(result)
    print(f"  Prompt {i+1}: {result['avg_time']:.3f}s ({result['tokens_per_sec']:.1f} tok/s)")

fp32_avg = np.mean([r['tokens_per_sec'] for r in fp32_results])
print(f"  Average: {fp32_avg:.1f} tokens/sec")

del model_fp32
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Benchmark FP16
print("\nBenchmarking FP16...")
model_fp16, _ = load_model_in_precision(model_name, torch.float16, device)

fp16_results = []
for i, prompt in enumerate(test_prompts):
    result = benchmark_inference(model_fp16, tokenizer, prompt, num_tokens=num_tokens_generate)
    fp16_results.append(result)
    print(f"  Prompt {i+1}: {result['avg_time']:.3f}s ({result['tokens_per_sec']:.1f} tok/s)")

fp16_avg = np.mean([r['tokens_per_sec'] for r in fp16_results])
print(f"  Average: {fp16_avg:.1f} tokens/sec")
print(f"  Speedup vs FP32: {fp16_avg / fp32_avg:.2f}x")

del model_fp16
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Benchmark BF16
if bf16_supported or device == "cpu":
    print("\nBenchmarking BF16...")
    model_bf16, _ = load_model_in_precision(model_name, torch.bfloat16, device)
    
    bf16_results = []
    for i, prompt in enumerate(test_prompts):
        result = benchmark_inference(model_bf16, tokenizer, prompt, num_tokens=num_tokens_generate)
        bf16_results.append(result)
        print(f"  Prompt {i+1}: {result['avg_time']:.3f}s ({result['tokens_per_sec']:.1f} tok/s)")
    
    bf16_avg = np.mean([r['tokens_per_sec'] for r in bf16_results])
    print(f"  Average: {bf16_avg:.1f} tokens/sec")
    print(f"  Speedup vs FP32: {bf16_avg / fp32_avg:.2f}x")
    
    del model_bf16
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
else:
    bf16_results = None
    bf16_avg = None

In [ ]:
# Visualize speed comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Tokens per second
precisions = ['FP32', 'FP16']
avg_speeds = [fp32_avg, fp16_avg]

if bf16_results:
    precisions.append('BF16')
    avg_speeds.append(bf16_avg)

x = np.arange(len(precisions))
bars = axes[0].bar(x, avg_speeds, color=['coral', 'lightblue', 'lightgreen'][:len(precisions)])

for bar, speed in zip(bars, avg_speeds):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{speed:.1f}\n({speed/avg_speeds[0]:.2f}x)',
                ha='center', va='bottom', fontweight='bold')

axes[0].set_ylabel('Tokens/Second', fontsize=12)
axes[0].set_title('Inference Speed by Precision', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(precisions, fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Per-prompt comparison
prompt_names = [f'Prompt {i+1}' for i in range(len(test_prompts))]
x = np.arange(len(prompt_names))
width = 0.25

fp32_speeds = [r['tokens_per_sec'] for r in fp32_results]
fp16_speeds = [r['tokens_per_sec'] for r in fp16_results]

axes[1].bar(x - width, fp32_speeds, width, label='FP32', color='coral')
axes[1].bar(x, fp16_speeds, width, label='FP16', color='lightblue')

if bf16_results:
    bf16_speeds = [r['tokens_per_sec'] for r in bf16_results]
    axes[1].bar(x + width, bf16_speeds, width, label='BF16', color='lightgreen')

axes[1].set_ylabel('Tokens/Second', fontsize=12)
axes[1].set_title('Speed Comparison Across Prompts', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(prompt_names, fontsize=10)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('speed_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Quality Impact Assessment

In [ ]:
def evaluate_quality(model, tokenizer, prompts: List[str], max_new_tokens: int = 50):
    """Generate text and evaluate quality."""
    outputs = []
    
    for prompt in prompts:
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        
        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # Greedy decoding for deterministic comparison
                use_cache=True,
            )
        
        generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        outputs.append(generated_text)
    
    return outputs


print("Evaluating generation quality across precisions...\n")
print("="*80)

eval_prompts = [
    "Artificial intelligence will transform",
    "The key to successful deep learning is",
]

# Generate with FP32
print("\nGenerating with FP32...")
model_fp32, _ = load_model_in_precision(model_name, torch.float32, device)
fp32_outputs = evaluate_quality(model_fp32, tokenizer, eval_prompts)
del model_fp32
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# Generate with FP16
print("Generating with FP16...")
model_fp16, _ = load_model_in_precision(model_name, torch.float16, device)
fp16_outputs = evaluate_quality(model_fp16, tokenizer, eval_prompts)
del model_fp16
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# Generate with BF16
if bf16_supported or device == "cpu":
    print("Generating with BF16...")
    model_bf16, _ = load_model_in_precision(model_name, torch.bfloat16, device)
    bf16_outputs = evaluate_quality(model_bf16, tokenizer, eval_prompts)
    del model_bf16
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
else:
    bf16_outputs = None

print("\nDone!")

In [ ]:
# Compare outputs
print("\n" + "="*80)
print("Quality Comparison (Greedy Decoding)")
print("="*80)

for i, prompt in enumerate(eval_prompts):
    print(f"\n{'='*80}")
    print(f"Prompt {i+1}: '{prompt}'")
    print(f"{'='*80}")
    
    print(f"\nFP32 Output:")
    print(f"  {fp32_outputs[i]}")
    
    print(f"\nFP16 Output:")
    print(f"  {fp16_outputs[i]}")
    
    if bf16_outputs:
        print(f"\nBF16 Output:")
        print(f"  {bf16_outputs[i]}")
    
    # Check if outputs are identical (greedy decoding should be)
    fp16_match = fp32_outputs[i] == fp16_outputs[i]
    print(f"\nFP16 matches FP32: {fp16_match}")
    
    if bf16_outputs:
        bf16_match = fp32_outputs[i] == bf16_outputs[i]
        print(f"BF16 matches FP32: {bf16_match}")

print("\n" + "="*80)
print("\nKey Findings:")
print("1. With greedy decoding, outputs are often identical or very similar")
print("2. Minor differences may occur due to rounding in attention/softmax")
print("3. For production: quality differences are typically negligible")
print("4. BF16 is preferred for large models (better numerical stability)")

## 5. Hardware Considerations

Different GPU architectures have different mixed precision capabilities.

In [ ]:
# Hardware capabilities overview
hardware_table = pd.DataFrame([
    {'Architecture': 'Volta (V100)', 'Compute': '7.0', 'FP32': 'Yes', 'FP16': 'Yes', 'BF16': 'No', 'Notes': 'First with Tensor Cores'},
    {'Architecture': 'Turing (T4, RTX 20)', 'Compute': '7.5', 'FP32': 'Yes', 'FP16': 'Yes', 'BF16': 'No', 'Notes': 'Improved INT8'},
    {'Architecture': 'Ampere (A100, RTX 30)', 'Compute': '8.0+', 'FP32': 'Yes', 'FP16': 'Yes', 'BF16': 'Yes', 'Notes': 'BF16 support added'},
    {'Architecture': 'Ada (RTX 40)', 'Compute': '8.9', 'FP32': 'Yes', 'FP16': 'Yes', 'BF16': 'Yes', 'Notes': 'Faster FP16/BF16'},
    {'Architecture': 'Hopper (H100)', 'Compute': '9.0', 'FP32': 'Yes', 'FP16': 'Yes', 'BF16': 'Yes', 'Notes': 'FP8 support'},
])

print("GPU Architecture Mixed Precision Support")
print("="*100)
print(hardware_table.to_string(index=False))

print("\n" + "="*100)
print("\nRecommendations by Hardware:")
print("  - Volta/Turing (Compute < 8.0): Use FP16")
print("  - Ampere+ (Compute >= 8.0): Use BF16 for large models, FP16 for small models")
print("  - Hopper (Compute 9.0+): Consider FP8 quantization (Stage 2)")
print("\nWhy BF16 for large models?")
print("  - Same exponent range as FP32 (better numerical stability)")
print("  - Less risk of overflow/underflow")
print("  - Recommended for 7B+ parameter models")

## 6. Production Best Practices

In [ ]:
def get_recommended_dtype(model_size_b: float, compute_capability: tuple) -> torch.dtype:
    """
    Get recommended dtype based on model size and GPU capability.
    
    Args:
        model_size_b: Model size in billions of parameters
        compute_capability: Tuple (major, minor) compute capability
    
    Returns:
        Recommended torch dtype
    """
    major, minor = compute_capability
    
    # BF16 requires Ampere or newer (compute capability >= 8.0)
    supports_bf16 = major >= 8
    
    if model_size_b >= 7.0 and supports_bf16:
        return torch.bfloat16
    elif major >= 7:  # Volta or newer
        return torch.float16
    else:
        return torch.float32


# Example recommendations
test_configs = [
    {'model': 'GPT-2 (124M)', 'size_b': 0.124, 'gpu': 'RTX 2080 Ti', 'cc': (7, 5)},
    {'model': 'GPT-2 XL (1.5B)', 'size_b': 1.5, 'gpu': 'V100', 'cc': (7, 0)},
    {'model': 'Llama 2 7B', 'size_b': 7.0, 'gpu': 'A100', 'cc': (8, 0)},
    {'model': 'Llama 2 7B', 'size_b': 7.0, 'gpu': 'RTX 4090', 'cc': (8, 9)},
    {'model': 'Llama 2 13B', 'size_b': 13.0, 'gpu': 'H100', 'cc': (9, 0)},
]

print("Precision Recommendations for Production")
print("="*100)
print(f"{'Model':<20} {'GPU':<15} {'Compute':<10} {'Recommended':<15} {'Reason':<30}")
print("-"*100)

for config in test_configs:
    dtype = get_recommended_dtype(config['size_b'], config['cc'])
    dtype_str = str(dtype).split('.')[-1]
    
    if dtype == torch.bfloat16:
        reason = "Large model + BF16 support"
    elif dtype == torch.float16:
        reason = "FP16 support available"
    else:
        reason = "Fallback to FP32"
    
    print(f"{config['model']:<20} {config['gpu']:<15} {str(config['cc']):<10} {dtype_str:<15} {reason:<30}")

print("\n" + "="*100)
print("\nProduction Checklist:")
checklist = [
    "✓ Check GPU compute capability before choosing precision",
    "✓ Use BF16 for 7B+ models on Ampere or newer GPUs",
    "✓ Use FP16 for smaller models or older GPUs",
    "✓ Always benchmark both speed and quality",
    "✓ Monitor for numerical instability (NaN, Inf)",
    "✓ Combine with KV caching for maximum speedup",
    "✓ Test on representative data before deployment",
    "✓ Document precision choice in deployment configs",
]

for item in checklist:
    print(f"  {item}")

In [ ]:
# Production code example
production_code = '''
# Production inference code with automatic precision selection

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model_for_production(model_name: str, device: str = "cuda"):
    """Load model with optimal precision for production."""
    
    # Determine optimal dtype
    if device == "cuda":
        compute_capability = torch.cuda.get_device_capability(0)
        
        # BF16 for Ampere+ GPUs, FP16 otherwise
        if compute_capability[0] >= 8:
            dtype = torch.bfloat16
            print(f"Using BF16 (GPU supports it)")
        else:
            dtype = torch.float16
            print(f"Using FP16 (BF16 not supported)")
    else:
        dtype = torch.float32
        print("Using FP32 (CPU mode)")
    
    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    
    return model, tokenizer


def generate_text(model, tokenizer, prompt: str, max_new_tokens: int = 50):
    """Generate text with production settings."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            use_cache=True,  # Always use KV cache
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# Usage
if __name__ == "__main__":
    model, tokenizer = load_model_for_production("gpt2")
    
    prompt = "The future of AI is"
    output = generate_text(model, tokenizer, prompt)
    
    print(f"Prompt: {prompt}")
    print(f"Output: {output}")
'''

print("Production Inference Code Template:")
print("="*100)
print(production_code)
print("="*100)
print("\nThis code automatically selects the best precision for your hardware!")

## Summary and Key Takeaways

### Performance Results
- **Memory Reduction**: 2x compared to FP32 (both FP16 and BF16)
- **Speed Improvement**: 1.5-2x on modern GPUs with Tensor Cores
- **Quality Impact**: Minimal - often identical outputs with greedy decoding

### FP16 vs BF16
- **FP16**: More precision bits, smaller range, better for small models
- **BF16**: Same range as FP32, better numerical stability, better for large models (7B+)
- **Hardware**: BF16 requires Ampere (compute capability 8.0+) or newer

### When to Use Each Precision
1. **FP32**: Only if accuracy is critical or older hardware
2. **FP16**: 
   - Small to medium models (< 7B parameters)
   - Volta/Turing GPUs (compute capability 7.x)
   - When you need maximum precision
3. **BF16**:
   - Large models (7B+ parameters)
   - Ampere/Ada/Hopper GPUs (compute capability 8.0+)
   - Recommended for production

### Production Recommendations
1. Auto-detect GPU capabilities and choose precision accordingly
2. Default to BF16 on Ampere+ GPUs
3. Always combine with KV caching (Notebook 10)
4. Benchmark both speed and quality before deployment
5. Monitor for numerical issues (NaN, Inf)

### Expected Improvements
- **Memory**: 2x reduction (enables larger batches or longer sequences)
- **Speed**: 1.5-2x faster (varies by GPU and model)
- **Throughput**: 1.5-2x more tokens/second
- **Cost**: Significant reduction in GPU costs for production

### Next Steps
- **Stage 1**: Static Batching (Notebook 12) - Increase throughput
- **Stage 1**: torch.compile (Notebook 13) - Additional 1.2-1.5x speedup
- **Stage 1**: INT8 Quantization (Notebook 14) - Further memory reduction

### References
- LLM_INFERENCE_ROADMAP.md - Stage 1: Basic Optimizations
- Mixed Precision Training: https://arxiv.org/abs/1710.03740
- BF16 for Deep Learning: https://arxiv.org/abs/1905.12322
- HuggingFace Performance Guide: https://huggingface.co/docs/transformers/performance